# Checkpoint 2 Notebook: Research Questions and Initial Methods

This notebook extends checkpoint 1 with additional exploratory analysis, question formation, and first-pass modeling experiments.

In [ ]:
# Reproducibility Header (Run this first)
# Run order note: use "Run All" from top to ensure reproducible outputs.

RANDOM_SEED = 42

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from preprocessing import preprocess_atp_matches

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR = "data/csv_data"
YEARS = list(range(2000, 2027))
CSV_PATHS = [os.path.join(DATA_DIR, f"atp_{year}.csv") for year in YEARS]

print(f"Random seed: {RANDOM_SEED}")
print(f"Number of input files configured: {len(CSV_PATHS)}")
print("Run order: Run All from top")


## 1) Project Scope Recap

- **Domain**: ATP men's professional tennis match outcomes.
- **Target goal**: build interpretable and predictive analyses around match winners and contextual factors.
- **Data source**: yearly ATP CSV files (`data/csv_data/atp_YYYY.csv`).

> **Decision Log (Scope):** We keep the same domain and dataset family from checkpoint 1 to ensure continuity and reduce integration risk; this checkpoint focuses on stronger RQ design and methodological grounding rather than changing data provenance.

In [ ]:
# Load and combine yearly ATP match files
frames = []
for path in CSV_PATHS:
    if os.path.exists(path):
        tmp = pd.read_csv(path)
        tmp["source_file"] = os.path.basename(path)
        frames.append(tmp)

if not frames:
    raise FileNotFoundError("No ATP CSV files were found under data/csv_data/.")

df = pd.concat(frames, ignore_index=True)
raw_shape = df.shape

# Reuse centralized preprocessing script for consistent cleaning decisions.
df = preprocess_atp_matches(df)

print(f"Rows before preprocessing: {raw_shape[0]:,}")
print(f"Rows after preprocessing : {len(df):,}")
print(f"Columns: {df.shape[1]}")
df.head(3)


## 2) Additional EDA for RQ Formation

> **Decision Log (EDA design):** We selected lightweight, high-signal EDA slices (missingness, rank distributions, and surface-level win context) because they directly inform feasible questions and candidate feature sets without overfitting to one tournament or year.

In [ ]:
# Quick structural EDA
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct.head(15)

In [ ]:
# Rank-focused EDA using confirmed singles ranking columns
rank_cols = [
    c for c in [
        "PlayerTeam1.SglRollRank",
        "PlayerTeam2.SglRollRank",
        "PlayerTeam1.SglRaceRank",
        "PlayerTeam2.SglRaceRank",
    ]
    if c in df.columns
]
if rank_cols:
    display(df[rank_cols].describe().T)
else:
    print("Singles ranking columns not found in this dataset version.")


In [ ]:
# Round distribution for context
round_col = "Round.ShortName"
if round_col in df.columns:
    plt.figure(figsize=(9, 4))
    order = df[round_col].value_counts().index
    sns.countplot(data=df, x=round_col, order=order)
    plt.title("Match count by round")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print(f"Column '{round_col}' not found.")


## 3) Research Questions

1. **RQ1 (Course: Large-Scale ML)**: How well can pre-match structured attributes predict whether a higher-ranked player wins?
2. **RQ2 (Course: Large-Scale ML + stratified analysis)**: How does predictive signal differ across court surfaces (Hard, Clay, Grass, Carpet)?
3. **RQ3 (External technique)**: Does an **Elo-style temporal rating feature** improve upset prediction over static rank-based features?

> **Decision Log (RQ definition):** RQ1 and RQ2 are directly aligned with course topics (large-scale ML); RQ3 introduces an external sequential rating approach that is not part of the listed course weeks while still feasible with current data.


## 4) Motivation & Feasibility

- **Motivation:** Accurate and interpretable pre-match outcome modeling is useful for understanding competitive structure in ATP events.
- **Feasibility:** The dataset is longitudinal, structured, and large enough for train/test evaluation plus subgroup analysis by surface.
- **Risk controls:** We explicitly avoid leakage fields (e.g., winner/loser-specific post-match stats) when creating features.

> **Decision Log (Feasibility):** We chose binary framing (higher-ranked wins vs. upset) as a tractable first target that supports both baseline and ensemble methods with straightforward evaluation metrics.

## 5) Methodological Plan

Planned workflow:
1. Build a binary label `higher_rank_wins` from winner/loser ranks.
2. Engineer pre-match features (rank and points gaps, categorical context like surface/round).
3. Train a regularized Logistic Regression baseline and a Random Forest nonlinear comparator (**course-aligned methods**).
4. Evaluate with Accuracy and ROC-AUC; inspect feature importance/proxy importance.
5. Repeat evaluation stratified by surface for RQ2.
6. Build an **external Elo-difference feature** from chronological matches and test whether it adds incremental predictive value.

> **Decision Log (Methods):** Logistic Regression and Random Forest were selected for course alignment and interpretability/performance trade-offs; Elo was added as an external temporal-strength method to satisfy checkpoint constraints.


In [ ]:
# Build target and features for initial runs using verified columns only
working = df.copy()
required = [
    "WinningPlayerId",
    "PlayerTeam1.PlayerId",
    "PlayerTeam2.PlayerId",
    "PlayerTeam1.SglRollRank",
    "PlayerTeam2.SglRollRank",
]
for col in required:
    if col not in working.columns:
        raise KeyError(f"Required column missing: {col}")

working = working.dropna(subset=required).copy()

# Define whether the better (numerically lower) pre-match rank won.
winner_is_team1 = working["WinningPlayerId"] == working["PlayerTeam1.PlayerId"]
winner_rank = np.where(winner_is_team1, working["PlayerTeam1.SglRollRank"], working["PlayerTeam2.SglRollRank"])
loser_rank = np.where(winner_is_team1, working["PlayerTeam2.SglRollRank"], working["PlayerTeam1.SglRollRank"])

working["winner_rank"] = pd.to_numeric(winner_rank, errors="coerce")
working["loser_rank"] = pd.to_numeric(loser_rank, errors="coerce")
working = working.dropna(subset=["winner_rank", "loser_rank"]).copy()

working["higher_rank_wins"] = (working["winner_rank"] < working["loser_rank"]).astype(int)
working["rank_gap"] = working["loser_rank"] - working["winner_rank"]

# Optional race-rank analogue to points-based gap.
if "PlayerTeam1.SglRaceRank" in working.columns and "PlayerTeam2.SglRaceRank" in working.columns:
    winner_race_rank = np.where(
        winner_is_team1,
        pd.to_numeric(working["PlayerTeam1.SglRaceRank"], errors="coerce"),
        pd.to_numeric(working["PlayerTeam2.SglRaceRank"], errors="coerce"),
    )
    loser_race_rank = np.where(
        winner_is_team1,
        pd.to_numeric(working["PlayerTeam2.SglRaceRank"], errors="coerce"),
        pd.to_numeric(working["PlayerTeam1.SglRaceRank"], errors="coerce"),
    )
    working["race_rank_gap"] = loser_race_rank - winner_race_rank
else:
    working["race_rank_gap"] = np.nan

candidate_features = ["rank_gap", "race_rank_gap", "Round.ShortName", "NumberOfSets"]
feature_cols = [c for c in candidate_features if c in working.columns]

X = working[feature_cols]
y = working["higher_rank_wins"]

num_features = [c for c in feature_cols if pd.api.types.is_numeric_dtype(X[c])]
cat_features = [c for c in feature_cols if c not in num_features]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),
    ],
    remainder="drop"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Features used:", feature_cols)


## 6) Initial Method Runs

> **Decision Log (Initial hyperparameters):**
> - Logistic Regression: `C=1.0`, `max_iter=1000`, `class_weight='balanced'` for a stable, regularized baseline under possible class imbalance.
> - Random Forest: `n_estimators=300`, `max_depth=12`, `min_samples_leaf=5`, `class_weight='balanced_subsample'` to reduce overfitting while preserving nonlinear flexibility.

In [ ]:
# Model 1: Logistic Regression
log_reg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
])

log_reg.fit(X_train, y_train)
log_pred = log_reg.predict(X_test)
log_prob = log_reg.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("  Accuracy:", round(accuracy_score(y_test, log_pred), 4))
print("  ROC-AUC :", round(roc_auc_score(y_test, log_prob), 4))

In [ ]:
# Model 2: Random Forest
rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest")
print("  Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("  ROC-AUC :", round(roc_auc_score(y_test, rf_prob), 4))

In [ ]:
# Round-level slice for RQ2 (illustrative)
round_col = "Round.ShortName"
if round_col in X_test.columns:
    results = []
    eval_df = X_test.copy()
    eval_df["y_true"] = y_test.values
    eval_df["log_prob"] = log_prob
    eval_df["rf_prob"] = rf_prob

    for round_name, g in eval_df.groupby(round_col):
        if g["y_true"].nunique() < 2:
            continue
        log_auc = roc_auc_score(g["y_true"], g["log_prob"])
        rf_auc = roc_auc_score(g["y_true"], g["rf_prob"])
        results.append((round_name, len(g), log_auc, rf_auc))

    round_perf = pd.DataFrame(results, columns=["round", "n", "log_auc", "rf_auc"]).sort_values("n", ascending=False)
    display(round_perf)
else:
    print("Round column not available for stratified evaluation.")


## 6b) External Method Feasibility Run (Elo Rating)

> **Decision Log (External feasibility):** We implement a lightweight Elo updater to verify feasibility of the external method and quantify whether chronological strength signals improve predictive power over static rank features.


In [ ]:
# External method prototype: Elo feature engineering and incremental evaluation
elo_df = working.copy()

date_col = "StartDate"
if date_col in elo_df.columns:
    elo_df = elo_df.sort_values(date_col).copy()

for req in ["WinningPlayerId", "PlayerTeam1.PlayerId", "PlayerTeam2.PlayerId"]:
    if req not in elo_df.columns:
        raise KeyError(f"Required column missing for Elo run: {req}")

player_elo = {}
def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

elo_diff = []
K = 24
BASE = 1500

for row in elo_df.itertuples(index=False):
    winner_id = row.__getattribute__("WinningPlayerId")
    p1_id = row.__getattribute__("PlayerTeam1.PlayerId")
    p2_id = row.__getattribute__("PlayerTeam2.PlayerId")

    if winner_id == p1_id:
        w, l = p1_id, p2_id
    else:
        w, l = p2_id, p1_id

    rw = player_elo.get(w, BASE)
    rl = player_elo.get(l, BASE)
    elo_diff.append(rw - rl)

    ew = expected_score(rw, rl)
    el = 1 - ew
    player_elo[w] = rw + K * (1 - ew)
    player_elo[l] = rl + K * (0 - el)

elo_df["elo_diff"] = elo_diff

external_features = [c for c in ["rank_gap", "race_rank_gap", "elo_diff", "Round.ShortName", "NumberOfSets"] if c in elo_df.columns]
X_ext = elo_df[external_features]
y_ext = elo_df["higher_rank_wins"]

num_ext = [c for c in external_features if pd.api.types.is_numeric_dtype(X_ext[c])]
cat_ext = [c for c in external_features if c not in num_ext]

ext_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_ext),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_ext),
    ],
    remainder="drop"
)

X_ext_train, X_ext_test, y_ext_train, y_ext_test = train_test_split(
    X_ext, y_ext, test_size=0.2, random_state=RANDOM_SEED, stratify=y_ext
)

ext_model = Pipeline(steps=[
    ("preprocess", ext_preprocess),
    ("model", LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
])

ext_model.fit(X_ext_train, y_ext_train)
ext_prob = ext_model.predict_proba(X_ext_test)[:, 1]
ext_auc = roc_auc_score(y_ext_test, ext_prob)

print("External Elo-Augmented Logistic Regression")
print("  Features:", external_features)
print("  ROC-AUC :", round(ext_auc, 4))


## 7) RQ-to-Method Mapping Table

| Research Question | Technique type | Primary algorithm(s) | Evaluation criteria |
|---|---|---|---|
| RQ1: Predict higher-ranked win | **Course** (Large-Scale ML) | Logistic Regression, Random Forest | Accuracy, ROC-AUC |
| RQ2: Surface differences | **Course** (Large-Scale ML, subgroup analysis) | Surface-stratified ROC-AUC | Stratified ROC-AUC + support size |
| RQ3: Elo feature value | **External** (temporal rating) | Elo rating updates + Logistic Regression | ΔROC-AUC vs baseline |

> **Decision Log (Mapping):** The table now explicitly marks course vs. external methods and ties each RQ to measurable outputs required by the rubric.


## 8) Collaboration Declaration

- Team members collaborated on scope alignment, feature brainstorming, and method selection.
- Notebook assembly and baseline implementation were completed jointly, with shared review before submission.
- All contributors reviewed the final RQs and method mapping for consistency.

> **Decision Log (Collaboration):** We used role separation (EDA lead, modeling lead, reviewer) to reduce duplication and improve quality control under checkpoint time constraints.